In [ ]:
import os
import json
import numpy as np
from sentence_transformers import SentenceTransformer
from keybert import KeyBERT
from transformers import pipeline


print("Loading models...")

# Embedding model
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")

# Keyword extraction
kw_model = KeyBERT(model="all-MiniLM-L6-v2")

# Correct summarization pipeline
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

device = 0 if torch.cuda.is_available() else -1

model_name = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

summarizer = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=device
)
# Sentiment model
sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

print("Models loaded successfully.\n")


# FUNCTIONS


def segment_text(sentences, block_size=5, k=1.0):
    """
    Segment transcript based on semantic similarity
    """
    embeddings = sbert_model.encode(sentences)

    blocks = []
    for i in range(0, len(embeddings), block_size):
        blocks.append(np.mean(embeddings[i:i+block_size], axis=0))

    similarities = []
    for i in range(len(blocks) - 1):
        sim = np.dot(blocks[i], blocks[i + 1]) / (
            np.linalg.norm(blocks[i]) * np.linalg.norm(blocks[i + 1])
        )
        similarities.append(sim)

    similarities = np.array(similarities)

    if len(similarities) == 0:
        return []

    threshold = similarities.mean() - k * similarities.std()

    return [i for i, sim in enumerate(similarities) if sim < threshold]


def extract_keywords(text, top_n=6):
    kws = kw_model.extract_keywords(
        text,
        keyphrase_ngram_range=(1, 2),
        stop_words="english",
        top_n=top_n
    )
    return [k[0] for k in kws]

def summarize_segment(text):

    words = text.split()

    if len(words) < 50:
        return text

    try:
        # Safe truncation for BART (~1024 tokens limit)
        truncated_text = " ".join(words[:900])

        inputs = tokenizer(
            truncated_text,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        )

        summary_ids = model.generate(
            inputs["input_ids"],
            max_length=200,
            min_length=60,
            num_beams=4,
            early_stopping=True
        )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True
        )

        return summary.strip()

    except Exception as e:
        print("Summarization error:", e)
        return text

def get_sentiment_score(text):
    result = sentiment_model(text[:512])[0]
    score = result["score"]

    if result["label"] == "NEGATIVE":
        return -round(score, 3)
    else:
        return round(score, 3)


def get_embedding(text):
    return sbert_model.encode(text).tolist()



# PROCESS ALL FILES

input_folder = "datasets/transcripts"
output_folder = "final_outputs"

os.makedirs(output_folder, exist_ok=True)

files = [f for f in os.listdir(input_folder) if f.endswith(".txt")]

for file in files:

    print("Processing:", file)

    episode_id = file.replace(".txt", "")
    duration = 1800  # Replace with actual duration if available

    with open(os.path.join(input_folder, file), "r", encoding="utf-8") as f:
        text = f.read()

    sentences = [s.strip() for s in text.split("\n") if s.strip()]

    boundaries = segment_text(sentences, block_size=5, k=1.0)

    segments_json = []
    segment_id = 1
    start_idx = 0
    current_time = 0.0
    total_sentences = len(sentences)
    block_size = 5

    for b in boundaries:

        end_idx = min((b + 1) * block_size - 1, total_sentences - 1)
        segment_text_block = " ".join(sentences[start_idx:end_idx + 1])

        start_time = current_time
        end_time = start_time + len(segment_text_block.split()) * 0.45

        segments_json.append({
            "segment_id": segment_id,
            "start_time": round(start_time, 2),
            "end_time": round(end_time, 2),
            "text": segment_text_block,
            "keywords": extract_keywords(segment_text_block),
            "summary": summarize_segment(segment_text_block),
            "sentiment_score": get_sentiment_score(segment_text_block)
        })

        current_time = end_time
        start_idx = end_idx + 1
        segment_id += 1

    # Final segment
    if start_idx < total_sentences:
        segment_text_block = " ".join(sentences[start_idx:])
        start_time = current_time
        end_time = start_time + len(segment_text_block.split()) * 0.45

        segments_json.append({
            "segment_id": segment_id,
            "start_time": round(start_time, 2),
            "end_time": round(end_time, 2),
            "text": segment_text_block,
            "keywords": extract_keywords(segment_text_block),
            "summary": summarize_segment(segment_text_block),
            "sentiment_score": get_sentiment_score(segment_text_block)
        })

    # Build Final JSON


    episode_data = {
        "episode_id": episode_id,
        "duration": duration,
        "segments": segments_json
    }

    # Save episode JSON
    with open(os.path.join(output_folder, f"{episode_id}.json"), "w", encoding="utf-8") as f:
        json.dump(episode_data, f, indent=2, ensure_ascii=False)

    # Save embeddings separately
    embedding_store = {
        seg["segment_id"]: get_embedding(seg["text"])
        for seg in segments_json
    }

    with open(os.path.join(output_folder, f"{episode_id}_embeddings.json"), "w") as f:
        json.dump(embedding_store, f)

    print(f"Finished {episode_id}\n")

print("All files processed successfully.")

Loading models...


Loading weights: 100%|██████████████████████████████████| 103/103 [00:00<00:00, 507.03it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████████████████████████████| 103/103 [00:00<00:00, 324.57it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Please make sure the generation config includes `forced_bos_token_id=0`. 
Loading weights: 100%|██| 

Models loaded successfully.

Processing: audio1.txt


Task 2 — Store Segment Embeddings for Search

In [11]:
from sentence_transformers import SentenceTransformer

sbert_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 183.39it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
def get_embedding(text):
    return sbert_model.encode(text).tolist()

In [13]:
embedding_store = {
    str(seg["segment_id"]): get_embedding(seg["text"])
    for seg in episode_data["segments"]
}

In [14]:
import json
import os

os.makedirs("final_outputs", exist_ok=True)

embedding_data = {
    "episode_id": episode_data["episode_id"],
    "embeddings": embedding_store
}

with open(f"final_outputs/{episode_data['episode_id']}_embeddings.json", "w") as f:
    json.dump(embedding_data, f)

print(" Embeddings saved successfully")

 Embeddings saved successfully


Task 3 — Freeze Final JSON Output Schema

In [23]:
def build_segment(
    segment_id,
    start_time,
    end_time,
    text,
    keywords,
    summary,
    sentiment_score
):
    return {
        "segment_id": int(segment_id),
        "start_time": float(round(start_time, 2)),
        "end_time": float(round(end_time, 2)),
        "text": str(text),
        "keywords": list(keywords),
        "summary": str(summary),
        "sentiment_score": float(sentiment_score)
    }

In [24]:
episode_data = {
    "episode_id": str(episode_id),
    "duration": float(duration),
    "segments": []
}

In [25]:
def validate_schema(episode_data):

    required_root_keys = {"episode_id", "duration", "segments"}
    required_segment_keys = {
        "segment_id",
        "start_time",
        "end_time",
        "text",
        "keywords",
        "summary",
        "sentiment_score"
    }

    # Check root keys
    assert set(episode_data.keys()) == required_root_keys, \
        "Root schema mismatch!"

    # Check segment keys
    for seg in episode_data["segments"]:
        assert set(seg.keys()) == required_segment_keys, \
            f"Segment schema mismatch in segment {seg['segment_id']}"

    print(" Schema validation passed")

Task 4 — Validate Timestamp Alignment

In [26]:
def validate_timestamps(episode_data, allow_small_gap=True, tolerance=0.01):

    segments = episode_data["segments"]
    duration = episode_data["duration"]

    assert len(segments) > 0, "No segments found."

    # Check first segment
    assert segments[0]["start_time"] >= 0, \
        "First segment starts before 0."

    for i, seg in enumerate(segments):

        start = seg["start_time"]
        end = seg["end_time"]

        # 1️Check positive duration
        assert start < end, \
            f"Invalid time range in segment {seg['segment_id']}"

        # 2️Check within episode duration
        assert end <= duration + tolerance, \
            f"Segment {seg['segment_id']} exceeds episode duration"

        # Check overlap with previous
        if i > 0:
            prev_end = segments[i-1]["end_time"]

            assert start >= prev_end - tolerance, \
                f"Overlap detected at segment {seg['segment_id']}"

            #  Optional gap detection
            if not allow_small_gap:
                assert abs(start - prev_end) <= tolerance, \
                    f"Gap detected before segment {seg['segment_id']}"

    print("Timestamp validation passed successfully.")